In [3]:
# train_cnc_01_model.py

import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


DATA_PATH = "/home/santhosh/AMDA/datasets/CNC_01_dataset.csv"
MODEL_DIR = "/home/santhosh/AMDA/models/CNC_01"
MODEL_PATH = f"{MODEL_DIR}/CNC_01_failure_model.pkl"

FEATURE_COLS = [
    "vibration_mm_s",
    "spindle_current_percent",
    "spindle_temperature_C",
    "acoustic_dba",
    "lubrication_pressure_bar",
    "coolant_temperature_C",
]

CLIP_RANGES = {
    "vibration_mm_s": (0.1, 8.0),
    "spindle_current_percent": (0, 160),
    "spindle_temperature_C": (20, 90),
    "acoustic_dba": (55, 110),
    "lubrication_pressure_bar": (0, 4),
    "coolant_temperature_C": (18, 50),
}


def add_realistic_noise(df, noise_level=0.04):
    np.random.seed(42)
    df = df.copy()

    for col in FEATURE_COLS:
        df[col] += df[col] * np.random.uniform(-noise_level, noise_level, len(df))
        df[col] += np.random.normal(0, df[col].std() * 0.03, len(df))
        low, high = CLIP_RANGES[col]
        df[col] = df[col].clip(low, high)

    return df


def train():
    os.makedirs(MODEL_DIR, exist_ok=True)

    df = pd.read_csv(DATA_PATH)
    df = df.dropna(subset=FEATURE_COLS + ["active_failure"])

    print("Class distribution:")
    print(df["active_failure"].value_counts())

    df = add_realistic_noise(df)

    X = df[FEATURE_COLS]
    y = df["active_failure"]

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
    )

    model = RandomForestClassifier(
        n_estimators=600,
        max_depth=16,
        min_samples_split=6,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
    print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    print(confusion_matrix(y_test, y_pred))

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y_encoded, cv=cv, scoring="f1_macro", n_jobs=-1)
    print("CV Macro F1:", cv_scores.mean())

    importance = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    importance.to_csv(f"{MODEL_DIR}/CNC_01_feature_importance.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.barh(importance["feature"], importance["importance"])
    plt.gca().invert_yaxis()
    plt.title("CNC_01 Feature Importance")
    plt.tight_layout()
    plt.savefig(f"{MODEL_DIR}/CNC_01_feature_importance.png")
    plt.close()

    joblib.dump({
        "model": model,
        "label_encoder": encoder,
        "feature_cols": FEATURE_COLS,
        "classes": list(encoder.classes_),
        "machine_type": "cnc",
        "machine_id": "CNC_01",
    }, MODEL_PATH)

    print("Saved:", MODEL_PATH)


if __name__ == "__main__":
    train()

Class distribution:
active_failure
none                       1020
tool_wear                  1007
spindle_bearing_failure     993
lubrication_failure         992
tool_breakage               988
Name: count, dtype: int64
Accuracy: 0.9513333333333334
Macro F1: 0.9515310117168865
Weighted F1: 0.9512690788939652
                         precision    recall  f1-score   support

    lubrication_failure       0.98      0.96      0.97       298
                   none       0.93      0.93      0.93       306
spindle_bearing_failure       0.95      0.97      0.96       298
          tool_breakage       0.98      0.99      0.98       296
              tool_wear       0.92      0.91      0.91       302

               accuracy                           0.95      1500
              macro avg       0.95      0.95      0.95      1500
           weighted avg       0.95      0.95      0.95      1500

[[287   0   5   1   5]
 [  1 285   3   2  15]
 [  5   0 288   2   3]
 [  0   0   2 293   1]
 [  1  22

## **Testing against custom test cases**

## **Model Accuracy and Confusion Matrix**

In [10]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# X_test and y_test should come from train_test_split
y_pred = model.predict(X_test_scaled)

print("Real test accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Real test accuracy: 0.991

Classification Report:
                         precision    recall  f1-score   support

    lubrication_failure       1.00      1.00      1.00       200
                   none       0.96      1.00      0.98       200
spindle_bearing_failure       1.00      1.00      1.00       200
          tool_breakage       1.00      1.00      1.00       200
              tool_wear       1.00      0.95      0.98       200

               accuracy                           0.99      1000
              macro avg       0.99      0.99      0.99      1000
           weighted avg       0.99      0.99      0.99      1000


Confusion Matrix:
[[200   0   0   0   0]
 [  0 200   0   0   0]
 [  0   0 200   0   0]
 [  0   0   0 200   0]
 [  0   9   0   0 191]]


## **Testing against custom test cases**

In [5]:
import pandas as pd
import joblib


MODEL_PATH = "/home/santhosh/AMDA/models/CNC_01/CNC_01_failure_model.pkl"

artifact = joblib.load(MODEL_PATH)

model = artifact["model"]
label_encoder = artifact["label_encoder"]
feature_cols = artifact["feature_cols"]


test_cases = [
    {
        "case": "Healthy baseline",
        "expected": "none",
        "vibration_mm_s": 0.82,
        "spindle_current_percent": 43.5,
        "spindle_temperature_C": 41.8,
        "acoustic_dba": 78.5,
        "lubrication_pressure_bar": 2.08,
        "coolant_temperature_C": 26.4,
    },
    {
        "case": "Healthy but slightly loaded",
        "expected": "none",
        "vibration_mm_s": 0.95,
        "spindle_current_percent": 47.0,
        "spindle_temperature_C": 44.0,
        "acoustic_dba": 81.0,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 27.0,
    },
    {
        "case": "Early tool wear",
        "expected": "tool_wear",
        "vibration_mm_s": 1.22,
        "spindle_current_percent": 53.0,
        "spindle_temperature_C": 46.0,
        "acoustic_dba": 84.0,
        "lubrication_pressure_bar": 2.05,
        "coolant_temperature_C": 27.2,
    },
    {
        "case": "Moderate tool wear",
        "expected": "tool_wear",
        "vibration_mm_s": 1.48,
        "spindle_current_percent": 58.0,
        "spindle_temperature_C": 49.0,
        "acoustic_dba": 87.0,
        "lubrication_pressure_bar": 2.00,
        "coolant_temperature_C": 28.0,
    },
    {
        "case": "Spindle bearing failure",
        "expected": "spindle_bearing_failure",
        "vibration_mm_s": 2.7,
        "spindle_current_percent": 51.0,
        "spindle_temperature_C": 58.0,
        "acoustic_dba": 88.0,
        "lubrication_pressure_bar": 2.03,
        "coolant_temperature_C": 27.5,
    },
    {
        "case": "Advanced spindle bearing failure",
        "expected": "spindle_bearing_failure",
        "vibration_mm_s": 3.8,
        "spindle_current_percent": 56.0,
        "spindle_temperature_C": 64.0,
        "acoustic_dba": 93.0,
        "lubrication_pressure_bar": 1.98,
        "coolant_temperature_C": 28.2,
    },
    {
        "case": "Lubrication failure",
        "expected": "lubrication_failure",
        "vibration_mm_s": 2.4,
        "spindle_current_percent": 52.0,
        "spindle_temperature_C": 59.0,
        "acoustic_dba": 87.5,
        "lubrication_pressure_bar": 0.95,
        "coolant_temperature_C": 27.8,
    },
    {
        "case": "Critical lubrication failure",
        "expected": "lubrication_failure",
        "vibration_mm_s": 3.2,
        "spindle_current_percent": 58.0,
        "spindle_temperature_C": 66.0,
        "acoustic_dba": 92.0,
        "lubrication_pressure_bar": 0.72,
        "coolant_temperature_C": 28.0,
    },
    {
        "case": "Tool breakage",
        "expected": "tool_breakage",
        "vibration_mm_s": 5.2,
        "spindle_current_percent": 77.0,
        "spindle_temperature_C": 49.0,
        "acoustic_dba": 93.5,
        "lubrication_pressure_bar": 2.04,
        "coolant_temperature_C": 27.5,
    },
    {
        "case": "Severe tool breakage",
        "expected": "tool_breakage",
        "vibration_mm_s": 6.4,
        "spindle_current_percent": 84.0,
        "spindle_temperature_C": 51.0,
        "acoustic_dba": 97.0,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 28.1,
    },
    {
        "case": "Borderline none vs tool wear",
        "expected": "tool_wear",
        "vibration_mm_s": 1.08,
        "spindle_current_percent": 50.0,
        "spindle_temperature_C": 45.0,
        "acoustic_dba": 82.5,
        "lubrication_pressure_bar": 2.07,
        "coolant_temperature_C": 27.0,
    },
    {
        "case": "Borderline tool wear vs bearing",
        "expected": "spindle_bearing_failure",
        "vibration_mm_s": 2.05,
        "spindle_current_percent": 55.0,
        "spindle_temperature_C": 53.5,
        "acoustic_dba": 86.5,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 27.4,
    },
]


df_test = pd.DataFrame(test_cases)

X_test = df_test[feature_cols]

pred_encoded = model.predict(X_test)
pred_labels = label_encoder.inverse_transform(pred_encoded)

probabilities = model.predict_proba(X_test)

df_test["predicted"] = pred_labels
df_test["confidence"] = probabilities.max(axis=1).round(4)
df_test["correct"] = df_test["expected"] == df_test["predicted"]

# Add top-2 predictions
top2_indices = probabilities.argsort(axis=1)[:, -2:][:, ::-1]

df_test["top_1"] = [
    label_encoder.classes_[idxs[0]] for idxs in top2_indices
]

df_test["top_1_confidence"] = [
    round(probabilities[i][idxs[0]], 4)
    for i, idxs in enumerate(top2_indices)
]

df_test["top_2"] = [
    label_encoder.classes_[idxs[1]] for idxs in top2_indices
]

df_test["top_2_confidence"] = [
    round(probabilities[i][idxs[1]], 4)
    for i, idxs in enumerate(top2_indices)
]


print("\nModel Scenario Test Results:")
print(
    df_test[
        [
            "case",
            "expected",
            "predicted",
            "confidence",
            "top_1",
            "top_1_confidence",
            "top_2",
            "top_2_confidence",
            "correct",
        ]
    ]
)

accuracy = df_test["correct"].mean()
print(f"\nScenario Test Accuracy: {accuracy:.2%}")

df_test.to_csv(
    "/home/santhosh/AMDA/models/CNC_01/CNC_01_manual_scenario_tests.csv",
    index=False
)

print("\nSaved test results to:")
print("/home/santhosh/AMDA/models/CNC_01/CNC_01_manual_scenario_tests.csv")


Model Scenario Test Results:
                                case                 expected  \
0                   Healthy baseline                     none   
1        Healthy but slightly loaded                     none   
2                    Early tool wear                tool_wear   
3                 Moderate tool wear                tool_wear   
4            Spindle bearing failure  spindle_bearing_failure   
5   Advanced spindle bearing failure  spindle_bearing_failure   
6                Lubrication failure      lubrication_failure   
7       Critical lubrication failure      lubrication_failure   
8                      Tool breakage            tool_breakage   
9               Severe tool breakage            tool_breakage   
10      Borderline none vs tool wear                tool_wear   
11   Borderline tool wear vs bearing  spindle_bearing_failure   

                  predicted  confidence                    top_1  \
0                      none      1.0000                 

## **Testing with live data stream from Simulator** 

In [9]:
import json
import time
import requests
import joblib
import pandas as pd

API_URL = "http://localhost:3000/stream/CNC_01"
MODEL_PATH = "/home/santhosh/AMDA/models/CNC_01/CNC_01_failure_model.pkl"

bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
label_encoder = bundle["label_encoder"]
feature_cols = bundle["feature_cols"]


def predict(raw_data):
    X = pd.DataFrame([raw_data])[feature_cols].astype(float)

    pred_id = model.predict(X)[0]
    pred_label = label_encoder.inverse_transform([pred_id])[0]

    probs = model.predict_proba(X)[0]
    confidence = probs.max()

    top2_idx = probs.argsort()[-2:][::-1]
    top2 = [
        (label_encoder.classes_[idx], probs[idx])
        for idx in top2_idx
    ]

    return pred_label, confidence, top2


def run():
    print(f"🤖 CNC_01 inference pipeline connecting to {API_URL}")

    while True:
        try:
            response = requests.get(API_URL, stream=True, timeout=None)

            for line in response.iter_lines():
                if not line:
                    continue

                decoded = line.decode("utf-8")

                if not decoded.startswith("data:"):
                    continue

                raw_data = json.loads(decoded.replace("data:", "").strip())

                prediction, confidence, top2 = predict(raw_data)

                icon = "🟢" if prediction == "none" else "🔴"

                print(
                    f"{icon} {raw_data['timestamp']} | "
                    f"CNC_01 | "
                    f"Server: {raw_data.get('active_failure')} | "
                    f"ML: {prediction} | "
                    f"Confidence: {confidence:.4f}"
                )

                print(
                    f"   Top-2: {top2[0][0]} ({top2[0][1]:.4f}) | "
                    f"{top2[1][0]} ({top2[1][1]:.4f})"
                )

                if prediction != "none":
                    print(f"   ⚠️ Decision: Inspect CNC_01 for {prediction.replace('_', ' ')}\n")

        except Exception as e:
            print(f"❌ Error: {e}")
            print("🔁 Reconnecting in 3 seconds...\n")
            time.sleep(3)


if __name__ == "__main__":
    run()

🤖 CNC_01 inference pipeline connecting to http://localhost:3000/stream/CNC_01
🟢 2026-04-28T13:03:55.907Z | CNC_01 | Server: none | ML: none | Confidence: 0.9953
   Top-2: none (0.9953) | tool_wear (0.0047)
🟢 2026-04-28T13:03:56.907Z | CNC_01 | Server: none | ML: none | Confidence: 0.9959
   Top-2: none (0.9959) | tool_wear (0.0041)
🟢 2026-04-28T13:03:57.907Z | CNC_01 | Server: none | ML: none | Confidence: 0.9987
   Top-2: none (0.9987) | tool_wear (0.0013)
🟢 2026-04-28T13:03:58.907Z | CNC_01 | Server: none | ML: none | Confidence: 0.9924
   Top-2: none (0.9924) | tool_wear (0.0076)
🟢 2026-04-28T13:03:59.908Z | CNC_01 | Server: none | ML: none | Confidence: 0.9986
   Top-2: none (0.9986) | tool_wear (0.0014)
🟢 2026-04-28T13:04:00.908Z | CNC_01 | Server: none | ML: none | Confidence: 0.9991
   Top-2: none (0.9991) | tool_wear (0.0009)
🟢 2026-04-28T13:04:01.908Z | CNC_01 | Server: none | ML: none | Confidence: 0.9960
   Top-2: none (0.9960) | tool_wear (0.0033)
🟢 2026-04-28T13:04:02.908Z

KeyboardInterrupt: 

In [10]:
import pandas as pd



In [14]:
df = pd.read_csv("/home/santhosh/AMDA/datasets/CNC_02_dataset.csv")

In [15]:
df['active_failure'].value_counts()

active_failure
none                    1022
spindle_misalignment    1004
cutting_overload        1002
coolant_failure          988
chatter                  984
Name: count, dtype: int64